## Voiceflow API for llama classification

In [ ]:
!pip install -q unsloth transformers accelerate peft flask pyngrok

In [ ]:
from unsloth import FastLanguageModel
import torch

HF_MODEL_ID = "alicedr/memo-llama-3b-instruct-arabic"

print("Loading model from HuggingFace...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL_ID,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
model.eval()
print("loaded alicedr/memo-llama-3b-instruct-arabic successfully!")

In [ ]:
def classify_text(text: str):
    """
    Classify patient statement as trigger (1) or non-trigger (0).
    """
    prompt = (
        "### Instruction:\n"
        "Classify the following patient statement as 0 (Not Triggered) or 1 (Triggered).\n"
        "Respond with exactly one character: 0 or 1.\n\n"
        "### Statement:\n"
        f"{text}\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=1,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True
        )

        output_ids = output.sequences
        scores = output.scores[0][0]

        predicted_token = tokenizer.decode(
            output_ids[0, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

        token_0 = tokenizer.encode("0", add_special_tokens=False)[0]
        token_1 = tokenizer.encode("1", add_special_tokens=False)[0]

        probs = torch.softmax(scores, dim=-1)
        prob_0 = probs[token_0].item()
        prob_1 = probs[token_1].item()

    predicted_label = None
    for ch in predicted_token:
        if ch in ("0", "1"):
            predicted_label = int(ch)
            break

    if predicted_label is None:
        predicted_label = 0

    confidence = prob_0 if predicted_label == 0 else prob_1

    return {
        "label": predicted_label,
        "confidence": float(confidence),
        "prob_0": float(prob_0),
        "prob_1": float(prob_1)
    }

In [ ]:
# Testing model inference works before starting the API!
print("Testing classification:")
test_result = classify_text("I have a severe headache")
print(f"Test result: {test_result}")

# API endpoint for classification

allows VoiceFlow to call this notebook and classify patient responses using alicedr/memo-llama-3b-instruct-arabic

In [ ]:
from flask import Flask, request, jsonify
from threading import Thread
import time

app = Flask(__name__)

@app.route('/classify', methods=['POST'])
def classify():
    """
    API endpoint for classification.
    Expects JSON: {"text": "patient statement"}
    Returns JSON: {"label": 0/1, "confidence": float}
    """
    try:
        data = request.get_json()

        if not data or 'text' not in data:
            return jsonify({'error': 'Missing "text" field in request'}), 400

        text = data['text']
        result = classify_text(text)

        return jsonify({
            'success': True,
            'text': text,
            'label': result['label'],
            'confidence': result['confidence'],
            'prob_0': result['prob_0'],
            'prob_1': result['prob_1']
        })

    except Exception as e:
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500

@app.route('/health', methods=['GET'])
def health():
    """Health check endpoint"""
    return jsonify({'status': 'healthy', 'model': HF_MODEL_ID})

print("Flask app created")

In [ ]:
from pyngrok import ngrok

print("\n" + "="*60)
print("Get your free ngrok auth token from https://ngrok.com")
print("="*60)
NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start Flask in background
def run_flask():
    app.run(port=5000, debug=False)

flask_thread = Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Wait for Flask to start
time.sleep(2)

# Create ngrok tunnel
public_url = ngrok.connect(5000)

print("\n" + "="*60)
print("✓ API is running!")
print("="*60)
print(f"Public URL: {public_url}")
print(f"\nEndpoint: {public_url}/classify")
print("\n Keep this notebook running!")
print("="*60)

# Keep running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nShutting down...")